## Audiofile Processing

In [11]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import soundfile as sf
from scipy.signal import butter, filtfilt, hilbert, spectrogram
from scipy.ndimage import gaussian_filter1d

In [ ]:
# functions for signal processing
def extract_timestamp_from_filename(filename):
    base = filename.replace(".flac", "")
    date_time = base.split("_")[1]
    
    date_str, time_str = date_time.split("-")
    month = int(date_str[:2])
    day = int(date_str[2:4])
    year = int("20" + date_str[4:])  # 23 -> 2023
    hour = int(time_str[:2])
    minute = int(time_str[2:4])
    second = int(time_str[4:]) if len(time_str) > 4 else 0
    return pd.Timestamp(year, month, day, hour, minute, second)

def shear_stress_noise_correction(signal, file_timestamp, shear_stress_df):
    """ shear_stress_df must have a datetime index and a column 'tau'"""
    # find the closest shear stress value
    if file_timestamp not in shear_stress_df.index:
        # interpolate if exact timestamp not found
        tau = shear_stress_df['taum'].reindex(shear_stress_df.index.union([file_timestamp])).interpolate().loc[file_timestamp]
    else:
        tau = shear_stress_df.loc[file_timestamp, 'taum']
    # amplification factor based on year 
    if file_timestamp.year == 2023:
        amp_factor = 0.7549 * (tau ** 1.3541)
    else:
        amp_factor = 0.0009 * (tau ** 3.655)
    return signal * amp_factor, tau, amp_factor

def gain_compensation(data, gain):
    linear_gain = 10 ** (gain / 20)
    calibrated_data = data / linear_gain
    return calibrated_data

def time_avg_freq_spectrum(data, fs, nperseg=2048):
    f, t, Sxx = spectrogram(data, fs, nperseg=nperseg)
    # average spectrum over time
    cols = np.arange(len(t))
    spec_avg = Sxx[:, cols].mean(axis=1)
    spec_sm = gaussian_filter1d(spec_avg, 2) # smooth with gaussian filter; sigma=2
    return f, spec_sm

def bandpass_filter(data, fs, lowcut, highcut, order=4):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band') # get filter coefficients
    filtered_data = filtfilt(b, a, data)
    return filtered_data

def filtering_and_amplification(data, fs, lowcut, highcut):
    # first filter and amplification
    filt1 = bandpass_filter(data, fs, lowcut, highcut)
    filt2 = filt1 * 20 
    # second filter and amplification
    filt3 = bandpass_filter(filt2, fs, lowcut, highcut)
    filt_def = filt3 * 10
    return filt_def

def envelope_extraction(data):
    analytic_signal = hilbert(data)
    envelope = np.abs(analytic_signal)
    return envelope

def rectangular_waveform(envelope, gains, time_array):
    threshold = 2 # volts
    # first column of results is the time index of detected impulses
    rect_hilbert = pd.DataFrame({'time': time_array})
    # loop over channels, apply gain, and detect impulses
    for ch, gain in gains.items():
        amplified = envelope * gain
        # create binary array: 1 if in threshold, else 0
        binary = ((amplified > threshold)).astype(int)
        # convert to 5 V pulse
        rectangular = binary * 5
        rect_hilbert[f'ch{ch}'] = rectangular
    return rect_hilbert

def count_impulses(rectangular, time, gains):
    dt = 1.0 # target bin size in seconds
    # build bins
    time_bins = np.arange(time[0], time[-1] + dt, dt) # include last bin edge
    time_index = pd.to_timedelta(time_bins[:-1], unit="s") # exclude last edge for index
    # prepare output array
    results = pd.DataFrame(index=time_index)
    # loop over channels
    for ch, gain in gains.items():
        rect_signal = rectangular[f"ch{ch}"].values
        # detect rising edges: 0→5 transitions
        rising_edges = np.diff((rect_signal > 0).astype(int)) == 1
        edge_times = time[1:][rising_edges]
        # bin counts into 1 s intervals
        counts, _ = np.histogram(edge_times, bins=time_bins)
        results[f"impulses_ch{ch}"] = counts
    return results

def quantify_impacts(results, envelope, rectangular, time, gains, file_name):
    # 1. add the original envelope as 'hilbert' column
    env_series = pd.Series(envelope, index=pd.to_timedelta(time, unit="s"))
    results["hilbert"] = env_series.resample("1s").mean()

    # 2. add rectangular waveforms and envelopes (resampled to 1 Hz)
    for ch in gains.keys(): 
        rect_series = pd.Series(rectangular[f"ch{ch}"].values, index=pd.to_timedelta(time, unit="s"))
        results[f"rect_ch{ch}"] = rect_series.resample('1s').max() # use max to keep 5V pulses
        # add per channel envelopes (amplified by gain)
        env_series = pd.Series(envelope * gains[ch], index=pd.to_timedelta(time, unit="s"))
        results[f"env_ch{ch}"] = env_series.resample("1s").mean()

    # 3. define analysis window (entire recording)
    t0 = time.min()  # start of entire period
    t1 = time.max()  # end of entire period
    window = results

    # 4. compute summary statistics over entire period
    impulse_sums = window.filter(like="impulses_ch").sum() # sum impulses per channel over entire period
    # amplitude stats (full envelope during entire period)
    max_amp = window["hilbert"].max()
    mean_amp = window["hilbert"].mean()
    median_amp = window["hilbert"].median()

    # 5. parse file name for date and time 
    base_name = os.path.basename(file_name).replace('.flac', '')  # remove extension
    date_time_part = base_name.split('_')[1]  # get "MMDDYY-HHMMSS"
    date_part, time_part = date_time_part.split('-')  # split into date and time
    # format date: MMDDYY -> MM/DD/YY
    formatted_date = f"{date_part[:2]}/{date_part[2:4]}/{date_part[4:]}"
    # format time: HHMMSS -> HH:MM:SS
    formatted_time = f"{time_part[:2]}:{time_part[2:4]}:{time_part[4:]}"
    # combine date and time
    datetime_string = f"{formatted_date} {formatted_time}"
    # collect results
    summary = {
        "time": datetime_string,
        "max_amp": max_amp,
        "mean_amp": mean_amp,
        "median_amp": median_amp
    }
    summary.update(impulse_sums.to_dict())  # add channel impulse counts

    # 6. calculate max, mean and median envelope only during impulses 
    for ch in gains.keys():
        impulse_mask = window[f"rect_ch{ch}"] > 0  # mask for times with impulses
        if impulse_mask.any():
            max_env = window.loc[impulse_mask, f"env_ch{ch}"].max()
            mean_env = window.loc[impulse_mask, f"env_ch{ch}"].mean()
            median_env = window.loc[impulse_mask, f"env_ch{ch}"].median()
        else:
            max_env = mean_env = median_env = 0  # no impulses detected
        summary[f"max_env_ch{ch}"] = max_env
        summary[f"mean_env_ch{ch}"] = mean_env
        summary[f"median_env_ch{ch}"] = median_env
    return summary

gains = {
    6: 1,
    5: 4,
    4: 16,
    3: 64,
    2: 256,
    1: 1024
}

def storm_processing(folder, preamp_gain, storm_number, preamp_channel, shear_stress_df):
    """Folder is a string, preamp_gain is in dB (int), 
    storm_number is a string for labeling and preamp_channel is 0 for left or 1 for right channel"""
    # determine if this folder is a calibration folder
    folder_lower = folder.lower()
    is_calibration = ("calibration" in folder_lower) or ("exp" in folder_lower)

    all_results = []  # List to store results from each file
    skipped_files = []  # Track files that couldn't be processed
    # loop through all the files in the folder
    for file in os.listdir(folder):
        if file.endswith('.flac'):
            file_path = os.path.join(folder, file)
            try:
                # extract timestamp from filename
                file_timestamp = extract_timestamp_from_filename(file)
                # read audio file
                raw_data, fs = sf.read(file_path)
                # select pipe channel (0 for left, 1 for right)
                selected_channel = raw_data[:, preamp_channel]
                # gain compensation
                data = gain_compensation(selected_channel, preamp_gain)

                # noise correction based on shear stress (returns amplified data)
                if not is_calibration:
                    amplified, tau, amp_factor = shear_stress_noise_correction(data, file_timestamp, shear_stress_df)
                else: # for calibration files, set tau and amp_factor to NaN
                    tau = np.nan
                    amp_factor = 1.0

                # create time vector
                time = np.arange(len(amplified)) / fs # time vector
                # get averaged frequency spectrum
                frequencies, spectrum = time_avg_freq_spectrum(amplified, fs)
                # filter and amplify signal
                filtered_data = filtering_and_amplification(amplified, fs, 700, 3000) 
                # extract envelope
                envelope = envelope_extraction(filtered_data)
                # rectangular waveform
                rectangular = rectangular_waveform(envelope, gains, time)
                # count impulses
                results = count_impulses(rectangular, time, gains)
                # quantify impacts and summarize metrics
                final = quantify_impacts(results, envelope, rectangular, time, gains, file_path)
                # add shear stress info
                final['tau'] = tau
                final['amp_factor'] = amp_factor
                final['timestamp'] = file_timestamp
                # append this file's results to the list
                all_results.append(final)
            except Exception as e:
                print(f"Error processing {file_path}: {e}")
                skipped_files.append(file_path)
                continue
    # convert list of dictionaries to DataFrame
    df_compiled = pd.DataFrame(all_results)
    # transpose to have metrics as index and files as columns
    df_compiled = df_compiled.set_index('time').T
    # Add storm number as metadata (optional)
    df_compiled.name = f"{storm_number}"
    return df_compiled

In [13]:
# import shear stress data
tau_2023 = pd.read_csv('mobile_shear_stress_2023.csv', parse_dates=['time'], index_col='time')
tau_2022 = pd.read_csv('mobile_shear_stress_2022.csv', parse_dates=['time'], index_col='time')
tau_2021 = pd.read_csv('mobile_shear_stress_2021.csv', parse_dates=['time'], index_col='time')

## Calibration Flood Experiments 

In [10]:
flood2 = storm_processing('Calibration/EXP-09-20-23-FLOW2/audio', 23, 'Flood_2', 0)

In [11]:
# processing the other flood experiments
flood3 = storm_processing('Calibration/EXP-09-20-23-FLOW3/audio', 23, 'Flood_3', 0)

In [12]:
flood4 = storm_processing('Calibration/EXP-09-21-23-FLOW4/audio', 23, 'Flood_4', 0)

In [13]:
flood5 = storm_processing('Calibration/EXP-09-21-23-FLOW5/audio', 23, 'Flood_5', 0)

In [14]:
flood6 = storm_processing('Calibration/EXP-09-22-23-FLOW6/audio', 30, 'Flood_6', 0)

In [15]:
flood8 = storm_processing('Calibration/EXP-09-22-23-FLOW8/audio', 30, 'Flood_8', 0)

In [17]:
flood2.to_csv('Calibration/flood_results/flood2_results.csv')
flood3.to_csv('Calibration/flood_results/flood3_results.csv')
flood4.to_csv('Calibration/flood_results/flood4_results.csv')
flood5.to_csv('Calibration/flood_results/flood5_results.csv')
flood6.to_csv('Calibration/flood_results/flood6_results.csv')
flood8.to_csv('Calibration/flood_results/flood8_results.csv')

## Summer Storms 

### Storm 1: 07/23/21

In [5]:
storm1 = storm_processing('H1/st1', 30, 'Storm_1', 0, tau_2021)

### Storm 2: 08/03/22

In [24]:
storm2 = storm_processing('H1/st2', 30, 'Storm_2', 1, tau_2022)
storm2.to_csv('Results/noise_corrected/2_amp_function_by_season/storm2_results.csv')

### Storm 3: 08/08/22

In [15]:
storm3 = storm_processing('H1/st3', 30, 'Storm_3', 1, tau_2022)
storm3.to_csv('Results/noise_corrected/2_amp_function_by_season/storm3_results.csv')

### Storm 4: 07/29/23

In [16]:
storm4 = storm_processing('H1/st4', 23, 'Storm_4', 0, tau_2023)
storm4.to_csv('Results/noise_corrected/2_amp_function_by_season/storm4_results.csv')

### Storm 5: 08/13/23

In [17]:
storm5 = storm_processing('H1/st5', 23, 'Storm_5', 0, tau_2023)
storm5.to_csv('Results/noise_corrected/2_amp_function_by_season/storm5_results.csv')

### Storm 6: 08/28/23

In [18]:
storm6 = storm_processing('H1/st6', 23, 'Storm_6', 0, tau_2023)
storm6.to_csv('Results/noise_corrected/2_amp_function_by_season/storm6_results.csv')

### Storm 7: 09/14/23

In [25]:
storm7 = storm_processing('H1/st7', 23, 'Storm_7', 0, tau_2023)
storm7.to_csv('Results/noise_corrected/2_amp_function_by_season/storm7_results.csv')

Export to CSV 

In [ ]:
#storm1.to_csv('Results/noise_corrected/storm1_results.csv')

# Spring 

### March

In [4]:
march = storm_processing('H1/march', 30, 'March', 0, tau_2023)
march.to_csv('Results/raw/march_results.csv')

### April

In [5]:
april1 = storm_processing('H1/april1', 30, 'April', 0, tau_2023)
april1.to_csv('Results/raw/april1_results.csv')

In [6]:
april2 = storm_processing('H1/april2', 30, 'April', 0, tau_2023)
april2.to_csv('Results/raw/april2_results.csv')

### May

In [4]:
may = storm_processing('H1/may', 30, 'May', 0, tau_2023)
may.to_csv('Results/raw/may_results.csv')